# 02 — PJM pricing node directory

Pulls the PJM `pnode` reference feed (Data Miner 2) and reduces it to the set of currently-active pricing nodes. Output: `data/processed/pnode_directory.parquet`.

This is the lookup that makes the plant-to-pnode join possible. Each of the 480 candidate plants in notebook 01 gets assigned to a pnode whose LMP represents the price that plant pays.

Reproducible: Restart + Run All re-pulls and rewrites the parquet. Requires `PJM_API_KEY` in `.env` (one live API call, no rate-limit pressure).

Notes on the raw feed, established empirically:
- The feed returns the full history of every node, so a single `pnode_id` appears multiple times with different effective/termination dates. We keep the row in effect today.
- Empty `zone` / `voltage_level` cells come back as a literal four-quote string, not blank. We normalize these to real nulls before saving.
- Active node count lands at ~14,450: ~13,967 `BUS` and ~483 `AGGREGATE`. No `LOCALE` nodes are currently active, a discrepancy with the original plan worth noting in docs/methods.md.

In [ ]:
import sys
import numpy as np
import pandas as pd
from pathlib import Path

# Resolve repo root by walking up to the folder containing both data/ and src/.
ROOT = Path.cwd()
while not ((ROOT / "data").is_dir() and (ROOT / "src").is_dir()) and ROOT != ROOT.parent:
    ROOT = ROOT.parent
print("repo root:", ROOT)
assert (ROOT / "data").is_dir(), "could not locate repo root containing data/"

# make src/ importable so we can use the PJM client module
src = ROOT / "src"
if str(src) not in sys.path:
    sys.path.insert(0, str(src))


## Pull the raw pnode directory

One call through the existing `PJMClient`. The `pnode` reference feed rejects `row_is_current` filtering, so we pull the full feed and reduce it ourselves below.

In [ ]:
from pjm_siting.pjm_api import PJMClient

client = PJMClient()
pnodes = client.pull_pnodes()
print("raw rows:", pnodes.shape)
print(pnodes.columns.tolist())
pnodes.head()


## Normalize junk empties and parse dates

The feed encodes empty `zone` / `voltage_level` cells as a literal quote-string. Turn those and any whitespace-only cells into real nulls, then parse the effective/termination dates so we can filter on them.

In [ ]:
# literal quote-strings and whitespace-only cells -> real null
pnodes_clean = pnodes.replace(r'^[\s"]*$', np.nan, regex=True)

pnodes_clean["effective_date"] = pd.to_datetime(pnodes_clean["effective_date"], errors="coerce")
pnodes_clean["termination_date"] = pd.to_datetime(pnodes_clean["termination_date"], errors="coerce")

print(pnodes_clean.dtypes)


## Reduce to currently-active nodes

Keep the row in effect today: termination date is null (still active) or in the future. If a node still has more than one active row, keep the latest by effective date.

In [ ]:
today = pd.Timestamp.today().normalize()
active = pnodes_clean[
    pnodes_clean["termination_date"].isna()
    | (pnodes_clean["termination_date"] > today)
].copy()

active = (
    active.sort_values("effective_date")
    .drop_duplicates("pnode_id", keep="last")
    .reset_index(drop=True)
)

print("raw rows:", len(pnodes), "-> active nodes:", len(active))
print(active["pnode_type"].value_counts())


## Validate before saving

In [ ]:
# sanity checks: the active set should be a fraction of raw, dominated by BUS
assert len(active) < len(pnodes), "active set did not shrink; termination filter failed"
assert active["pnode_id"].is_unique, "duplicate pnode_id in active set"
assert "BUS" in active["pnode_type"].values, "no BUS nodes; gen-bus mapping would be impossible"

print("active nodes:", len(active))
print("unique pnode_id:", active["pnode_id"].nunique())
print("\ntype breakdown:")
print(active["pnode_type"].value_counts())


## Save the directory

In [ ]:
out_dir = ROOT / "data" / "processed"
out_dir.mkdir(parents=True, exist_ok=True)
out_path = out_dir / "pnode_directory.parquet"

active.to_parquet(out_path, index=False)
print("saved", active.shape, "to", out_path)
